# Advanced Distribution Charts (Solution)

Use advanced distribution charts: ridgeline-like panels, beeswarm, raincloud-like combos.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "owid_co2_subset.csv").exists():
            return p
    raise FileNotFoundError("Cannot locate data/owid_co2_subset.csv")

root = resolve_repo_root()
df = pd.read_csv(root / "data" / "owid_co2_subset.csv")
df = df[df["iso_code"].astype(str).str.len() == 3].copy()
d_recent = df[df["year"] >= 2000].copy()
latest_year = (
    d_recent.dropna(subset=["co2_per_capita", "gdp", "population"])
    ["year"]
    .max()
)
d_latest = d_recent[d_recent["year"] == latest_year].copy()
d_latest.head()

## 1) Ridgeline-like via stacked KDE facets

In [ ]:
work = d_latest.dropna(subset=['co2_per_capita', 'gdp']).copy()
work['gdp_q'] = pd.qcut(work['gdp'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])

g = sns.FacetGrid(work, row='gdp_q', hue='gdp_q', aspect=6, height=1.1)
g.map(sns.kdeplot, 'co2_per_capita', fill=True, alpha=0.6)
g.map(sns.kdeplot, 'co2_per_capita', color='w', lw=1)
g.fig.subplots_adjust(hspace=-0.45)
g.set_titles(row_template='GDP {row_name}')
for ax in g.axes.flat:
    ax.set_yticks([])
    ax.set_ylabel('')
plt.show()

## 2) Beeswarm (strip + jitter)

In [ ]:
work = d_latest.dropna(subset=['co2_per_capita', 'gdp']).copy()
work['gdp_q'] = pd.qcut(work['gdp'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
plt.figure(figsize=(9,4))
sns.stripplot(data=work, x='gdp_q', y='co2_per_capita', hue='gdp_q', jitter=0.25, dodge=False, alpha=0.65)
plt.title('Beeswarm-like: CO2 per capita by GDP quantile')
plt.tight_layout(); plt.show()

## 3) Raincloud-like (violin + box + points)

In [ ]:
work = d_latest.dropna(subset=['co2_per_capita', 'gdp']).copy()
work['gdp_q'] = pd.qcut(work['gdp'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
fig, ax = plt.subplots(figsize=(9,4))
sns.violinplot(data=work, x='gdp_q', y='co2_per_capita', inner=None, color='lightgray', ax=ax)
sns.boxplot(data=work, x='gdp_q', y='co2_per_capita', width=0.2, showcaps=True, boxprops={'facecolor':'white'}, ax=ax)
sns.stripplot(data=work, x='gdp_q', y='co2_per_capita', color='black', size=2, alpha=0.35, ax=ax)
ax.set_title('Raincloud-like: CO2 per capita by GDP quantile')
plt.tight_layout(); plt.show()

## 4) Contour density plot (2D KDE)

Dùng khi muốn thấy vùng mật độ cao/thấp của hai biến liên tục mà scatter thường bị chồng điểm.

In [ ]:
plot_df = d_latest.dropna(subset=['gdp','co2_per_capita']).copy()
plot_df['log_gdp'] = np.log10(plot_df['gdp'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)

sns.scatterplot(data=plot_df, x='log_gdp', y='co2_per_capita', alpha=0.45, ax=axes[0])
axes[0].set_title('Scatter (overplot risk)')

sns.kdeplot(data=plot_df, x='log_gdp', y='co2_per_capita', fill=True, levels=8, thresh=0.05, cmap='mako', ax=axes[1])
sns.kdeplot(data=plot_df, x='log_gdp', y='co2_per_capita', levels=8, color='white', linewidths=0.7, ax=axes[1])
axes[1].set_title('Contour density (2D KDE)')

for ax in axes:
    ax.set_xlabel('log10(gdp)')
    ax.set_ylabel('co2_per_capita')

plt.tight_layout(); plt.show()

## Reflection
- Nêu 2 điểm học được về chart selection.
- Chỉ ra 1 rủi ro diễn giải sai với loại chart trong lab này.